In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
# Upload the train and test files
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [3]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score


# Feature Engineering
def feature_engineering(data):
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1  # +1 for the individual

    # Create a feature for whether the passenger is alone
    data['IsAlone'] = 1  # Default value
    data.loc[data['FamilySize'] > 1, 'IsAlone'] = 0  # 0 if family size > 1

    # Encode 'Sex' as numeric
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})

    # Fill missing values in 'Age' with the median age
    data['Age'] = data['Age'].fillna(data['Age'].median())  # Use fillna without inplace

    # Convert 'Embarked' to numeric
    data['Embarked'] = data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})
    data['Embarked'] = data['Embarked'].fillna(-1)  # Use fillna without inplace

    # Drop unnecessary columns
    data = data.drop(['Name', 'Ticket', 'Cabin'], axis=1)

    return data


# Apply feature engineering
train_data = feature_engineering(train_data)
y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch", "FamilySize", "IsAlone", "Age", "Embarked"]
X = train_data[features]


In [4]:
# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_leaf=2, min_samples_split=10, random_state=1)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
lr = LogisticRegression(max_iter=1000, random_state=42)

# Voting classifier ensemble using 'soft' voting for probability-based aggregation
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='soft'
)

# Train the model with the best parameters
voting_clf.fit(X_train, y_train)
y_val_pred = voting_clf.predict(X_val)

# Calculate accuracy on the validation set
accuracy = accuracy_score(y_val, y_val_pred)

# Print accuracy score
print(f"Accuracy on validation set: {accuracy:.4f}")

Accuracy on validation set: 0.8268


In [5]:
# Apply feature engineering
test_data = feature_engineering(test_data)
# Define features for the test set (ensure it matches training features)
test_features = ["Pclass", "Sex", "SibSp", "Parch", "FamilySize", "IsAlone", "Age", "Embarked"]
X_test = test_data[test_features]

voting_clf.fit(X, y)
predictions = voting_clf.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
